In [1]:
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "weather_project"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

def quality_report(collection_name):
    col = db[collection_name]

    total = col.count_documents({})
    print(f"\n----- QUALITÉ {collection_name} -----")
    print(f"Documents totaux : {total}")

    if total == 0:
        print("⚠ collection vide")
        return

    # Champs obligatoires
    required_fields = ["name", "location.latitude", "location.longitude"]

    missing_count = 0
    for f in required_fields:
        q = {f: {"$exists": False}}
        missing_count += col.count_documents(q)

    duplicates = len(list(col.aggregate([
        {"$group": {"_id": "$station_id", "count": {"$sum": 1}}},
        {"$match": {"count": {"$gt": 1}}}
    ])))

    print(f"Taux champs manquants : {round((missing_count/total)*100,2)} %")
    print(f"Station_id dupliqués : {duplicates}")

    bad_type = col.count_documents({"location.latitude": {"$type": "string"}})
    print(f"Latitude mauvais type : {bad_type}")

    quality_score = 100
    quality_score -= (missing_count/total)*20
    quality_score -= duplicates*10
    print(f"Score Qualité global : {round(max(0,quality_score),2)}/100")


if __name__ == "__main__":
    quality_report("la_madeleine")
    quality_report("ichtegem")
    quality_report("infoclimat")



----- QUALITÉ la_madeleine -----
Documents totaux : 1
Taux champs manquants : 0.0 %
Station_id dupliqués : 0
Latitude mauvais type : 0
Score Qualité global : 100.0/100

----- QUALITÉ ichtegem -----
Documents totaux : 1
Taux champs manquants : 0.0 %
Station_id dupliqués : 0
Latitude mauvais type : 0
Score Qualité global : 100.0/100

----- QUALITÉ infoclimat -----
Documents totaux : 4
Taux champs manquants : 0.0 %
Station_id dupliqués : 0
Latitude mauvais type : 0
Score Qualité global : 100.0/100
